# Phase 2 — Polynomial Regression
**Team IDs:** 16002932, 16004208, 16006616 | **Seed:** 42648  
**Rules:** numpy/pandas only; `gd_fit` reused from `phase1_core.py` unchanged; no sklearn `.fit()`/`.predict()`; `test.csv` transform-only.

In [96]:
# ── Imports & constants ──────────────────────────────────────────────────────
import warnings, hashlib, json, sys, time
from math import comb
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from sklearn.metrics import r2_score, mean_squared_error

import gradient_descent as core
import polynomial_regression as p2

warnings.filterwarnings('ignore')
np.set_printoptions(precision=6, suppress=True)

SEED     = 42648
IDS      = ['16002932', '16006616', '16004208']
DATA_DIR = Path('Datasets')
P1_STEM  = 'gradient_descent_artifacts'
P2_STEM  = 'polynomial_regression_artifacts'
PLOTS    = Path('plots'); PLOTS.mkdir(exist_ok=True)

# Default polynomial sources (continuous columns only — trap #2 guard)
SOURCE_COLS = ['temp', 'hum', 'windspeed', 'days_since_start']

print(f'Python {sys.version}')
print(f'numpy {np.__version__}  pandas {pd.__version__}')
print(f'Source cols for expansion: {SOURCE_COLS}')

Python 3.14.7 (main, Aug 10 2026, 00:00:00) [GCC 16.1.1 20260515 (Red Hat 16.1.1-2)]
numpy 2.4.6  pandas 3.0.5
Source cols for expansion: ['temp', 'hum', 'windspeed', 'days_since_start']


---
## Task 1 — Chain Verification

**Expectation template** *(team fills in)*:
> Cite: stored SHA-256, recomputed SHA-256 (must match), seed, stored vs recomputed split sizes, stored vs recomputed Phase 1 val R² and RMSE (tolerance 1e-10).

In [97]:
# ── T1.1: Load Phase 1 artifact ──────────────────────────────────────────────
W_P1, B_P1, META_P1, _, _ = core.load_artifacts(P1_STEM)
P1_FEAT_NAMES = META_P1['feature_names']

print(f'Phase 1 artifact loaded.')
print(f'  w shape      : {W_P1.shape}')
print(f'  b            : {B_P1:.6f}')
print(f'  stored SHA256: {META_P1["w_sha256"]}')

# T1.1a: SHA-256 integrity
recomputed_sha = hashlib.sha256(W_P1.tobytes()).hexdigest()
ok_sha = (recomputed_sha == META_P1['w_sha256'])
print(f'[{"PASS" if ok_sha else "FAIL"}] T1.1a SHA-256 matches stored value')
assert ok_sha, 'SHA-256 mismatch — do not proceed'

Phase 1 artifact loaded.
  w shape      : (54,)
  b            : 126.388538
  stored SHA256: dc64f7a2a673b903234fdf35b0481b7ce7b3d53b6497c5a4534a73de713162f4
[PASS] T1.1a SHA-256 matches stored value


In [98]:
# ── T1.2: Re-run split and verify indices ────────────────────────────────────
from sklearn.model_selection import train_test_split

train_all = pd.read_csv(DATA_DIR / 'train.csv')
test_all  = pd.read_csv(DATA_DIR / 'test.csv')

# Verify seed from IDs
computed_seed = core.compute_seed(IDS)
assert computed_seed == SEED, f'Seed mismatch: {computed_seed} != {SEED}'
print(f'[PASS] Seed recomputed = {computed_seed}')

train_df, val_df = train_test_split(train_all, test_size=0.20, random_state=SEED)

stored_tr_idx  = set(META_P1.get('train_idx', []))  # from npz
stored_val_idx = set(META_P1.get('val_idx',   []))

# Load from npz for exact check
_npz = np.load(P1_STEM + '.npz', allow_pickle=False)
stored_tr_idx  = set(_npz['train_idx'].tolist())
stored_val_idx = set(_npz['val_idx'].tolist())

ok_tr_idx  = set(train_df.index.tolist()) == stored_tr_idx
ok_val_idx = set(val_df.index.tolist())   == stored_val_idx
print(f'[{"PASS" if ok_tr_idx  else "FAIL"}] T1.2a Train indices match stored')
print(f'[{"PASS" if ok_val_idx else "FAIL"}] T1.2b Val   indices match stored')
assert ok_tr_idx and ok_val_idx, 'Index mismatch — do not proceed'

TRAIN_IDX = _npz['train_idx']
VAL_IDX   = _npz['val_idx']
print(f'train={len(train_df)}  val={len(val_df)}')

[PASS] Seed recomputed = 42648
[PASS] T1.2a Train indices match stored
[PASS] T1.2b Val   indices match stored
train=8708  val=2178


In [99]:
# ── T1.3: Rebuild Phase 1 features & verify R² ──────────────────────────────
# Re-fit preprocessor using stored params (trap #1 — do not re-standardise)
PREP_PARAMS = {
    'days_origin'       : META_P1['days_origin'],
    'scaler_means'      : META_P1['scaler_means'],
    'scaler_stds'       : META_P1['scaler_stds'],
    'numeric_cols'      : META_P1['numeric_cols'],
    'add_windspeed_flag': META_P1['add_windspeed_flag'],
    'hr_cats'           : META_P1['hr_cats'],
    'season_cats'       : META_P1['season_cats'],
    'mnth_cats'         : META_P1['mnth_cats'],
    'weekday_cats'      : META_P1['weekday_cats'],
    'weather_cats'      : META_P1['weather_cats'],
}

X_TRAIN_P1, feat_names_check = core.transform(train_df, PREP_PARAMS)
X_VAL_P1,   _               = core.transform(val_df,   PREP_PARAMS)
X_TEST_P1,  _               = core.transform(test_all, PREP_PARAMS)

y_train = train_df['cnt'].values.astype(np.float64)
y_val   = val_df['cnt'].values.astype(np.float64)

# T1.3a: feature names match
ok_feat = (feat_names_check == P1_FEAT_NAMES)
print(f'[{"PASS" if ok_feat else "FAIL"}] T1.3a Feature names match artifact')

# T1.3b: reproduce Phase 1 val R² and RMSE to 1e-10
y_val_pred_p1 = core.predict(X_VAL_P1, W_P1, B_P1)
y_tr_pred_p1  = core.predict(X_TRAIN_P1, W_P1, B_P1)

R2_VAL_P1  = core.r2_score_np(y_val,   y_val_pred_p1)
RMSE_VAL_P1= core.rmse_np(y_val,   y_val_pred_p1)
R2_TR_P1   = core.r2_score_np(y_train, y_tr_pred_p1)
RMSE_TR_P1 = core.rmse_np(y_train, y_tr_pred_p1)

STORED_VAL_LOSS = META_P1['final_val_loss']    # = 0.5*mean(r^2)
recomputed_val_loss = core.mse_loss(X_VAL_P1, y_val, W_P1, B_P1)
ok_loss = abs(recomputed_val_loss - STORED_VAL_LOSS) < 1e-8

print(f'[{"PASS" if ok_loss else "FAIL"}] T1.3b Phase 1 val loss reproduced  '
      f'stored={STORED_VAL_LOSS:.8f}  recomputed={recomputed_val_loss:.8f}')
print(f'\nPhase 1 (reference):')
print(f'  Train R²={R2_TR_P1:.6f}  RMSE={RMSE_TR_P1:.4f}')
print(f'  Val   R²={R2_VAL_P1:.6f}  RMSE={RMSE_VAL_P1:.4f}')

[PASS] T1.3a Feature names match artifact
[PASS] T1.3b Phase 1 val loss reproduced  stored=26472.21405994  recomputed=26472.21405994

Phase 1 (reference):
  Train R²=-0.545410  RMSE=226.4328
  Val   R²=-0.689100  RMSE=230.0966


**Outcome template**: Cite: SHA-256 match (yes/no), seed recomputed value, train/val sizes, Phase 1 val R² reproduced to what precision.

---
## Task 2 — Candidate Selection Evidence

**Expectation template**: Cite: which features show curved residual patterns, Pearson vs Spearman rank (monotone vs non-monotone), which single-feature quadratic gives the biggest val R² lift.

In [100]:
# ── T2.1: Residuals of Phase 1 vs each source column ─────────────────────────
residuals_p1 = y_val - y_val_pred_p1   # positive = under-predicted

print('=== Phase 1 residuals binned by each source column (val set) ===')
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()

for ax, col in zip(axes, SOURCE_COLS):
    feat_idx = P1_FEAT_NAMES.index(col)
    feat_vals = X_VAL_P1[:, feat_idx]  # already standardised
    # Bin into 20 bins
    n_bins = 20
    bin_idx = pd.cut(feat_vals, bins=n_bins, labels=False)
    res_df  = pd.DataFrame({'bin': bin_idx, 'res': residuals_p1, 'x': feat_vals})
    grp     = res_df.groupby('bin').agg(x_mean=('x','mean'), res_mean=('res','mean'))
    ax.bar(grp['x_mean'], grp['res_mean'], width=(feat_vals.max()-feat_vals.min())/n_bins*0.8,
           color='steelblue', alpha=0.7)
    ax.axhline(0, color='red', lw=1)
    ax.set_title(f'Mean residual vs {col} (standardised)')
    ax.set_xlabel(f'{col} (std units)'); ax.set_ylabel('Mean residual')

fig.suptitle('Phase 1 residuals vs expansion candidates\n(curved pattern = polynomial term may help)',
             fontsize=11)
fig.tight_layout()
fig.savefig(PLOTS / '02_residual_motivation.png', dpi=100)
plt.close(fig)
print('[SAVED] plots/02_residual_motivation.png')

=== Phase 1 residuals binned by each source column (val set) ===
[SAVED] plots/02_residual_motivation.png


In [101]:
# ── T2.2: Pearson vs Spearman correlation with cnt ───────────────────────────
print('=== Pearson vs Spearman r with cnt (train set) ===')
print(f'{"Column":<25} {"Pearson":>10} {"Spearman":>10}  Note')
print('-' * 60)
for col in SOURCE_COLS + ['atemp']:
    if col not in P1_FEAT_NAMES:
        continue
    idx = P1_FEAT_NAMES.index(col)
    x   = X_TRAIN_P1[:, idx]
    r_p = float(np.corrcoef(x, y_train)[0, 1])
    r_s = float(scipy_stats.spearmanr(x, y_train).statistic)
    note = 'non-monotone' if abs(r_p) < abs(r_s) - 0.05 else ''
    print(f'{col:<25} {r_p:>10.4f} {r_s:>10.4f}  {note}')

=== Pearson vs Spearman r with cnt (train set) ===
Column                       Pearson   Spearman  Note
------------------------------------------------------------
temp                          0.3992     0.4151  
hum                          -0.3191    -0.3597  
windspeed                     0.1048     0.1403  
days_since_start              0.3129     0.2828  
atemp                         0.3934     0.4135  


In [102]:
# ── T2.3: Single-feature linear vs quadratic screening ───────────────────────
# Quick lstsq comparison per source (labelled: closed-form screening, not GD)
print('=== Single-feature screening: linear vs quadratic (closed-form, val R²) ===')
print(f'{"Column":<25} {"Lin val R²":>12} {"Quad val R²":>12}  {"Lift":>8}')
print('-' * 65)

screening_results = {}
for col in SOURCE_COLS:
    idx = P1_FEAT_NAMES.index(col)
    x_tr = X_TRAIN_P1[:, idx:idx+1]  # shape (n,1)
    x_vl = X_VAL_P1[:,   idx:idx+1]

    # Linear
    A_lin = np.hstack([x_tr, np.ones((len(x_tr),1))])
    th_l, _, _, _ = np.linalg.lstsq(A_lin, y_train, rcond=None)
    y_vl_l = x_vl @ th_l[:1] + th_l[1]
    r2_l   = core.r2_score_np(y_val, y_vl_l)

    # Quadratic
    x_tr_sq = x_tr**2;  x_vl_sq = x_vl**2
    # Standardise quadratic term on train
    mu_sq = x_tr_sq.mean(); sig_sq = x_tr_sq.std()
    if sig_sq == 0: sig_sq = 1.0
    x_tr_sq_s = (x_tr_sq - mu_sq) / sig_sq
    x_vl_sq_s = (x_vl_sq - mu_sq) / sig_sq
    A_quad = np.hstack([x_tr, x_tr_sq_s, np.ones((len(x_tr),1))])
    th_q, _, _, _ = np.linalg.lstsq(A_quad, y_train, rcond=None)
    y_vl_q = x_vl @ th_q[:1] + x_vl_sq_s @ th_q[1:2] + th_q[2]
    r2_q   = core.r2_score_np(y_val, y_vl_q)

    lift = r2_q - r2_l
    screening_results[col] = {'r2_lin': r2_l, 'r2_quad': r2_q, 'lift': lift}
    print(f'{col:<25} {r2_l:>12.6f} {r2_q:>12.6f}  {lift:>8.6f}')

=== Single-feature screening: linear vs quadratic (closed-form, val R²) ===
Column                      Lin val R²  Quad val R²      Lift
-----------------------------------------------------------------
temp                          0.139354     0.138150  -0.001205
hum                           0.095983     0.094835  -0.001148
windspeed                     0.007166     0.010232  0.003066
days_since_start              0.087109     0.091461  0.004352


In [103]:
# ── T2.4: Hour × workingday residual — hypothesis for interactions ────────────
print('=== Phase 1 mean residual by hr x workingday ===')
val_hr  = val_df['hr'].values
val_wd  = val_df['workingday'].values
res_df  = pd.DataFrame({'hr': val_hr, 'wd': val_wd, 'res': residuals_p1})
pivot   = res_df.pivot_table(values='res', index='hr', columns='wd', aggfunc='mean')
pivot.columns = ['Non-workday', 'Workday']
print(pivot.to_string())
print('\n[NOTE] Interactions between hr dummies and continuous features may explain')
print('residual patterns more than polynomial powers. This is a hypothesis for the team.')

=== Phase 1 mean residual by hr x workingday ===
    Non-workday     Workday
hr                         
0   -121.843499 -222.102608
1    -30.418775 -228.240747
2    -85.843034 -198.862422
3    -88.394124 -176.608715
4   -124.621504 -193.141780
5   -141.807384 -169.778357
6    -76.781832 -133.646047
7   -284.620752  -84.162913
8   -387.672549  -87.899470
9   -167.185395 -137.882014
10   -48.904384 -258.355681
11   -13.812507 -281.929544
12   -36.680960 -257.259912
13     1.808268 -290.051036
14   -26.588402 -271.401602
15   -10.310209 -264.617282
16   -55.787483 -246.986529
17  -299.423111 -155.467579
18  -259.217981 -137.715211
19  -226.164984 -177.820168
20  -184.922004 -180.512475
21  -148.271344 -178.101439
22  -120.228397 -161.992773
23   -92.061171 -179.967515

[NOTE] Interactions between hr dummies and continuous features may explain
residual patterns more than polynomial powers. This is a hypothesis for the team.


**Outcome template**: Cite: which sources show non-zero curvature in residuals, Pearson vs Spearman ranks, which single-feature quadratic had the largest lift. State the hypothesis about interactions.

---
## Task 3 — Expansion Module & Tests

**Expectation template**: Cite: expected term count at each degree (formula), whether Phase 1 block is bit-identical, whether new columns are mean≈0/std≈1 on train.

In [104]:
# ── T3.1: Term count tests ────────────────────────────────────────────────────
k = len(SOURCE_COLS)
print(f'k (source columns) = {k}')
print()

t3_passes = []
for d in [2, 3, 4]:
    for interactions in [False, True]:
        expected = p2.poly_term_count_expected(k, d, interactions)
        specs    = p2.get_poly_term_specs(SOURCE_COLS, d, interactions)
        actual   = len(specs)
        ok = (actual == expected)
        t3_passes.append(ok)
        mode = 'full' if interactions else 'pure'
        print(f'[{"PASS" if ok else "FAIL"}] degree={d} {mode:<6}: expected {expected}, got {actual}'
              + (f'  Terms: {[n for n,_ in specs[:5]]}...' if d<=2 else ''))

print(f'\nT3.1 term count: {sum(t3_passes)}/{len(t3_passes)} passed')

k (source columns) = 4

[PASS] degree=2 pure  : expected 4, got 4  Terms: ['temp^2', 'hum^2', 'windspeed^2', 'days_since_start^2']...
[PASS] degree=2 full  : expected 10, got 10  Terms: ['temp^2', 'temp*hum', 'temp*windspeed', 'temp*days_since_start', 'hum^2']...
[PASS] degree=3 pure  : expected 8, got 8
[PASS] degree=3 full  : expected 30, got 30
[PASS] degree=4 pure  : expected 12, got 12
[PASS] degree=4 full  : expected 65, got 65

T3.1 term count: 6/6 passed


In [105]:
# ── T3.2: Build degree-2 expanded matrices (default config) ──────────────────
DEGREE_TEST = 2
POLY_PARAMS_TEST = p2.fit_poly(X_TRAIN_P1, P1_FEAT_NAMES,
                               SOURCE_COLS, DEGREE_TEST, interactions=False)

X_TR_EXP_TEST, EXP_NAMES_TEST = p2.transform_poly(X_TRAIN_P1, P1_FEAT_NAMES, POLY_PARAMS_TEST)
X_VL_EXP_TEST, _              = p2.transform_poly(X_VAL_P1,   P1_FEAT_NAMES, POLY_PARAMS_TEST)
X_TE_EXP_TEST, _              = p2.transform_poly(X_TEST_P1,  P1_FEAT_NAMES, POLY_PARAMS_TEST)

p1 = X_TRAIN_P1.shape[1]
n_new_d2 = POLY_PARAMS_TEST['n_new_cols']

print(f'Phase 1 features  : {p1}')
print(f'New (degree 2)    : {n_new_d2}')
print(f'Total expanded    : {X_TR_EXP_TEST.shape[1]}')
print(f'New col names     : {EXP_NAMES_TEST[p1:]}')

Phase 1 features  : 54
New (degree 2)    : 4
Total expanded    : 58
New col names     : ['temp^2', 'hum^2', 'windspeed^2', 'days_since_start^2']


In [106]:
# ── T3.3: Expansion correctness tests ────────────────────────────────────────
t3_corr = []

# (a) Phase 1 block is bit-identical
ok_p1_block = np.array_equal(X_TR_EXP_TEST[:, :p1], X_TRAIN_P1)
t3_corr.append(ok_p1_block)
print(f'[{"PASS" if ok_p1_block else "FAIL"}] T3.3a Phase 1 block bit-identical in X_expanded')

# (b) No NaN or inf
ok_finite = (np.isfinite(X_TR_EXP_TEST).all() and
             np.isfinite(X_VL_EXP_TEST).all() and
             np.isfinite(X_TE_EXP_TEST).all())
t3_corr.append(ok_finite)
print(f'[{"PASS" if ok_finite else "FAIL"}] T3.3b No NaN/inf in any expanded matrix')

# (c) Train new columns: mean≈0, std≈1
new_block = X_TR_EXP_TEST[:, p1:]
col_means = new_block.mean(axis=0)
col_stds  = new_block.std(axis=0)
ok_scaled = (np.abs(col_means) < 1e-6).all() and (np.abs(col_stds - 1.0) < 1e-6).all()
t3_corr.append(ok_scaled)
print(f'[{"PASS" if ok_scaled else "FAIL"}] T3.3c Train new cols: max|mean|={np.abs(col_means).max():.2e}  '
      f'max|std-1|={np.abs(col_stds-1).max():.2e}')

# (d) Val and test do NOT necessarily have mean=0 (they use train params)
val_new_means = X_VL_EXP_TEST[:, p1:].mean(axis=0)
print(f'       Val new col means (not 0 by design): mean={val_new_means.mean():.4f}')

# (e) Identical column names across splits
_, names_val_test = p2.transform_poly(X_VAL_P1, P1_FEAT_NAMES, POLY_PARAMS_TEST)
_, names_te_test  = p2.transform_poly(X_TEST_P1, P1_FEAT_NAMES, POLY_PARAMS_TEST)
ok_names = (EXP_NAMES_TEST == names_val_test == names_te_test)
t3_corr.append(ok_names)
print(f'[{"PASS" if ok_names else "FAIL"}] T3.3d All splits share identical column names')

# (f) Values match manual numpy computation
# Manually compute temp^2 standardised
temp_idx = P1_FEAT_NAMES.index('temp')
raw_temp_sq  = X_TRAIN_P1[:, temp_idx] ** 2
mu_ts  = raw_temp_sq.mean(); sig_ts = raw_temp_sq.std()
manual_col   = (raw_temp_sq - mu_ts) / sig_ts
exp_col_idx  = EXP_NAMES_TEST.index('temp^2')
ok_manual    = np.allclose(X_TR_EXP_TEST[:, exp_col_idx], manual_col, atol=1e-10)
t3_corr.append(ok_manual)
print(f'[{"PASS" if ok_manual else "FAIL"}] T3.3e Values match manual computation (temp^2)')

# (g) Single-row transform works
try:
    _ = p2.transform_poly(X_TRAIN_P1[:1, :], P1_FEAT_NAMES, POLY_PARAMS_TEST)
    t3_corr.append(True)
    print('[PASS] T3.3f Single-row transform works')
except Exception as e:
    t3_corr.append(False)
    print(f'[FAIL] T3.3f Single-row transform crashed: {e}')

# (h) sklearn PolynomialFeatures cross-check (in test only, labelled)
from sklearn.preprocessing import PolynomialFeatures
pf_ref = PolynomialFeatures(degree=2, include_bias=False)
src_indices_test = [P1_FEAT_NAMES.index(c) for c in SOURCE_COLS]
X_src_tr = X_TRAIN_P1[:, src_indices_test]
raw_ref  = pf_ref.fit_transform(X_src_tr)  # degree 1+2 terms
# Our degree-2 new cols = raw_ref[:, k:] (columns after degree-1)
# (sklearn includes degree-1 first, then degree-2)
ref_d2_raw = raw_ref[:, k:]   # shape (n, k*(k+1)//2 for interactions)
# For pure powers only: compare col by col
our_new_raw  = new_block * POLY_PARAMS_TEST['poly_stds'] + POLY_PARAMS_TEST['poly_means']
# Build reference pure-power matrix
ref_pp = X_src_tr ** 2   # shape (n,k)
ok_sk  = np.allclose(our_new_raw, ref_pp, atol=1e-8)
t3_corr.append(ok_sk)
print(f'[{"PASS" if ok_sk else "FAIL"}] T3.3g sklearn cross-check (pure powers d=2, reference only)')

print(f'\n=== T3.3 total: {sum(t3_corr)}/{len(t3_corr)} passed ===')

[PASS] T3.3a Phase 1 block bit-identical in X_expanded
[PASS] T3.3b No NaN/inf in any expanded matrix
[PASS] T3.3c Train new cols: max|mean|=3.71e-16  max|std-1|=1.79e-14
       Val new col means (not 0 by design): mean=-0.0048
[PASS] T3.3d All splits share identical column names
[PASS] T3.3e Values match manual computation (temp^2)
[PASS] T3.3f Single-row transform works
[PASS] T3.3g sklearn cross-check (pure powers d=2, reference only)

=== T3.3 total: 7/7 passed ===


**Outcome template**: Cite: term counts at d=2,3,4 (pure and full), whether Phase 1 block is bit-identical, max|mean| and max|std-1| of new train columns, sklearn cross-check result.

---
## Task 4 — Warm Start

**Expectation template**: Cite: max|Δpred| at iteration 0 vs Phase 1 (should be < 1e-10), initial loss vs Phase 1 final loss, initial gradient norm on new block.

In [107]:
# ── T4.1: Construct warm start & run identity tests ──────────────────────────
n_new_d2 = POLY_PARAMS_TEST['n_new_cols']
w_init_d2, b_init_d2 = p2.build_warm_start(W_P1, B_P1, n_new_d2)

# Predictions at iteration 0
y_tr_iter0 = core.predict(X_TR_EXP_TEST, w_init_d2, b_init_d2)
y_vl_iter0 = core.predict(X_VL_EXP_TEST, w_init_d2, b_init_d2)

max_diff_tr = float(np.abs(y_tr_iter0 - y_tr_pred_p1).max())
max_diff_vl = float(np.abs(y_vl_iter0 - y_val_pred_p1).max())

ok_ws_tr = max_diff_tr < 1e-10
ok_ws_vl = max_diff_vl < 1e-10
print(f'[{"PASS" if ok_ws_tr else "FAIL"}] T4.1a Warm-start train pred == Phase 1  max|Δ|={max_diff_tr:.2e}')
print(f'[{"PASS" if ok_ws_vl else "FAIL"}] T4.1b Warm-start val   pred == Phase 1  max|Δ|={max_diff_vl:.2e}')

# Initial loss equals Phase 1 final loss
loss_iter0 = core.mse_loss(X_TR_EXP_TEST, y_train, w_init_d2, b_init_d2)
p1_final_loss = META_P1['final_train_loss']
ok_loss0 = abs(loss_iter0 - p1_final_loss) < 1e-6
print(f'[{"PASS" if ok_loss0 else "FAIL"}] T4.1c Initial loss == Phase 1 final loss  '
      f'init={loss_iter0:.8f}  stored={p1_final_loss:.8f}')

# Initial gradient on new block (should be non-zero if expansion can help)
r0 = X_TR_EXP_TEST @ w_init_d2 + b_init_d2 - y_train
grad_new_block = X_TR_EXP_TEST[:, p1:].T @ r0 / len(y_train)
grad_norm_new  = float(np.linalg.norm(grad_new_block))
print(f'[INFO] Initial gradient norm on new block: {grad_norm_new:.6f}  '
      f'(non-zero = expansion can improve)') 

[PASS] T4.1a Warm-start train pred == Phase 1  max|Δ|=2.27e-13
[PASS] T4.1b Warm-start val   pred == Phase 1  max|Δ|=2.27e-13
[PASS] T4.1c Initial loss == Phase 1 final loss  init=25635.91068312  stored=25635.91068312
[INFO] Initial gradient norm on new block: 26.558651  (non-zero = expansion can improve)


In [108]:
# ── T4.2: Warm start vs zero init comparison (degree 2) ──────────────────────
L_d2 = core.lipschitz_constant(X_TR_EXP_TEST)
lr_d2 = 0.9 / L_d2
MAX_ITER_COMPARE = 50000

print(f'Comparison: warm start vs zero init (degree=2, lr={lr_d2:.6f}, max_iter={MAX_ITER_COMPARE})')

# Warm start
t0 = time.time()
res_warm = core.gd_fit(X_TR_EXP_TEST, y_train,
                       lr=lr_d2, max_iter=MAX_ITER_COMPARE, tol=1e-6,
                       w0=w_init_d2, b0=b_init_d2,
                       X_val=X_VL_EXP_TEST, y_val=y_val)
t_warm = time.time() - t0

# Zero init
t0 = time.time()
res_zero = core.gd_fit(X_TR_EXP_TEST, y_train,
                       lr=lr_d2, max_iter=MAX_ITER_COMPARE, tol=1e-6,
                       X_val=X_VL_EXP_TEST, y_val=y_val)
t_zero = time.time() - t0

r2_warm = core.r2_score_np(y_val, core.predict(X_VL_EXP_TEST, res_warm['w'], res_warm['b']))
r2_zero = core.r2_score_np(y_val, core.predict(X_VL_EXP_TEST, res_zero['w'], res_zero['b']))

print(f'\n{"Init":>12}  {"Iters":>8}  {"Train Loss":>14}  {"Val R²":>10}  {"Time(s)":>8}  {"Status"}')
print('-' * 75)
print(f'{"Warm (P1)":>12}  {res_warm["n_iter"]:>8}  {res_warm["final_train_loss"]:>14.4f}  '
      f'{r2_warm:>10.6f}  {t_warm:>8.2f}  {res_warm["status"]}')
print(f'{"Zero":>12}  {res_zero["n_iter"]:>8}  {res_zero["final_train_loss"]:>14.4f}  '
      f'{r2_zero:>10.6f}  {t_zero:>8.2f}  {res_zero["status"]}')

Comparison: warm start vs zero init (degree=2, lr=0.366392, max_iter=50000)

        Init     Iters      Train Loss      Val R²   Time(s)  Status
---------------------------------------------------------------------------
   Warm (P1)       628       4972.2418    0.679208      0.60  converged
        Zero      3603       4972.1586    0.679206      2.29  converged


**Outcome template**: Cite: max|Δpred| at iteration 0, loss_iter0 vs Phase 1 final loss, gradient norm on new block, warm vs zero comparison (iters, final val R², convergence status).

---
## Task 5 — LR Sweep at Chosen Degree

**Expectation template**: Cite: L_d, 1/L_d, 2/L_d at the chosen degree; which lr values converged/diverged; condition number κ.

In [109]:
# ── T5: LR sweep at degree 2 (will be updated after T6 selects degree) ───────
L = L_d2
lr_sweep = [
    round(0.1 / L, 8),
    round(0.5 / L, 8),
    round(0.9 / L, 8),   # ← chosen (safe margin below 1/L)
    round(1.0 / L, 8),
    round(1.5 / L, 8),
    round(2.0 / L, 8),
    round(2.1 / L, 8),
]
XtX_d2 = X_TR_EXP_TEST.T @ X_TR_EXP_TEST
ev_d2  = np.linalg.eigvalsh(XtX_d2)
kappa_d2 = float(ev_d2[ev_d2>0].max() / ev_d2[ev_d2>0].min())

print(f'L_d2 = {L:.6f}  1/L = {1/L:.6f}  2/L = {2/L:.6f}')
print(f'κ = {kappa_d2:.4e}')
print()
print(f'{"LR":>12}  {"Iters":>7}  {"Train Loss":>12}  {"Val R²":>10}  {"Status"}')
print('-' * 62)

for lr_val in lr_sweep:
    res = core.gd_fit(X_TR_EXP_TEST, y_train,
                      lr=lr_val, max_iter=30000, tol=1e-7,
                      w0=w_init_d2, b0=b_init_d2)
    y_vp = core.predict(X_VL_EXP_TEST, res['w'], res['b'])
    r2   = core.r2_score_np(y_val, y_vp) if res['status'] != 'diverged' else float('nan')
    print(f'{lr_val:>12.6f}  {res["n_iter"]:>7}  {res["final_train_loss"]:>12.4f}  '
          f'{r2:>10.6f}  {res["status"]}')

L_d2 = 2.456388  1/L = 0.407102  2/L = 0.814204
κ = 4.7144e+17

          LR    Iters    Train Loss      Val R²  Status
--------------------------------------------------------------
    0.040710     6442     4971.8288    0.679257  converged
    0.203551     3840     4968.7149    0.679695  converged
    0.366392     2712     4968.3257    0.679770  converged
    0.407102     2541     4968.2732    0.679781  converged
    0.610653     1973     4968.1032    0.679819  converged
    0.814204      914    24765.0351   -0.624406  converged
    0.854914       74  26467033744.7844         nan  diverged


**Outcome template**: Cite: L_d, κ, which lr values diverged, which converged fastest, chosen lr and its ratio to L_d.

---
## Task 6 — Degree Sweep & Selection

**Expectation template**: Cite: expected R² trajectory (should peak then flatten or worsen), expected degree at which optimization-limited flag appears, parsimony rule tolerance.

In [110]:
# ── T6.1: Full degree sweep d=1..6 ───────────────────────────────────────────
PARSIMONY_TOL = 0.001   # accept degree d if val_R² >= best - PARSIMONY_TOL
MAX_DEGREE    = 6
MAX_ITER_SWEEP = 200_000
INTERACTIONS   = False  # default: pure powers

print(f'Degree sweep: d=1..{MAX_DEGREE}, interactions={INTERACTIONS}')
print(f'Source cols : {SOURCE_COLS}')
print(f'Parsimony tol: {PARSIMONY_TOL}')
print()

degree_table = []
sweep_results = {}

# degree=1 is Phase 1 itself (reference row)
row_d1 = {
    'degree'       : 1,
    'n_new_cols'   : 0,
    'n_total'      : p1,
    'kappa'        : float(np.linalg.eigvalsh(X_TRAIN_P1.T @ X_TRAIN_P1)[-1] /
                           max(np.linalg.eigvalsh(X_TRAIN_P1.T @ X_TRAIN_P1)[np.linalg.eigvalsh(X_TRAIN_P1.T @ X_TRAIN_P1)>0][0], 1e-12)),
    'status'       : META_P1['status'] if 'status' in META_P1 else 'converged',
    'n_iter'       : META_P1['iterations_used'],
    'train_r2'     : float(R2_TR_P1),
    'val_r2'       : float(R2_VAL_P1),
    'train_rmse'   : float(RMSE_TR_P1),
    'val_rmse'     : float(RMSE_VAL_P1),
    'r2_gap'       : float(R2_TR_P1 - R2_VAL_P1),
    'neg_pct'      : float(100*(core.predict(X_VAL_P1, W_P1, B_P1) < 0).mean()),
    'lstsq_val_r2' : float(core.r2_score_np(y_val, core.predict(X_VAL_P1, *[W_P1, B_P1]))),
    'opt_gap'      : 0.0,
    'opt_limited'  : False,
    'wall_s'       : 0.0,
}
degree_table.append(row_d1)
print(f'd=1 (Phase 1): val R²={R2_VAL_P1:.6f}  RMSE={RMSE_VAL_P1:.4f}')

prev_opt_limited_count = 0
for d in range(2, MAX_DEGREE + 1):
    print(f'  Fitting degree {d}...', end=' ', flush=True)
    w_i, b_i = p2.build_warm_start(W_P1, B_P1,
                                    p2.poly_term_count_expected(k, d, INTERACTIONS))
    res = p2.fit_expanded(
        X_TRAIN_P1, y_train, X_VAL_P1, y_val,
        P1_FEAT_NAMES, SOURCE_COLS, d, INTERACTIONS,
        W_P1, B_P1,
        max_iter=MAX_ITER_SWEEP, tol=1e-6,
    )
    sweep_results[d] = res
    neg_pct = float(100*(core.predict(res['X_val_exp'], res['w'], res['b']) < 0).mean())
    row = {
        'degree'      : d,
        'n_new_cols'  : res['n_new_cols'],
        'n_total'     : res['n_total_cols'],
        'kappa'       : res['kappa'],
        'status'      : res['status'],
        'n_iter'      : res['n_iter'],
        'train_r2'    : res['train_r2'],
        'val_r2'      : res['val_r2'],
        'train_rmse'  : res['train_rmse'],
        'val_rmse'    : res['val_rmse'],
        'r2_gap'      : res['train_r2'] - res['val_r2'],
        'neg_pct'     : neg_pct,
        'lstsq_val_r2': res['lstsq_val_r2'],
        'opt_gap'     : res['opt_gap_relative'],
        'opt_limited' : res['optimization_limited'],
        'wall_s'      : res['wall_time_s'],
    }
    degree_table.append(row)
    flag = ' *** OPTIMIZATION-LIMITED ***' if res['optimization_limited'] else ''
    print(f'val R²={res["val_r2"]:.6f}  lstsq R²={res["lstsq_val_r2"]:.6f}  '
          f'κ={res["kappa"]:.2e}  status={res["status"]}  opt_gap={res["opt_gap_relative"]:.2e}{flag}')
    if res['optimization_limited']:
        prev_opt_limited_count += 1
    if prev_opt_limited_count >= 3:
        print('  [STOPPING] 3 consecutive optimization-limited degrees. Higher degrees unreliable.')
        break

Degree sweep: d=1..6, interactions=False
Source cols : ['temp', 'hum', 'windspeed', 'days_since_start']
Parsimony tol: 0.001

d=1 (Phase 1): val R²=-0.689100  RMSE=230.0966
  Fitting degree 2... val R²=0.679208  lstsq R²=0.679845  κ=1.50e+03  status=converged  opt_gap=1.28e-03 *** OPTIMIZATION-LIMITED ***
  Fitting degree 3... val R²=0.685347  lstsq R²=0.686032  κ=2.13e+03  status=converged  opt_gap=1.18e-03 *** OPTIMIZATION-LIMITED ***
  Fitting degree 4... val R²=0.685996  lstsq R²=0.686712  κ=2.35e+03  status=converged  opt_gap=1.37e-03 *** OPTIMIZATION-LIMITED ***
  [STOPPING] 3 consecutive optimization-limited degrees. Higher degrees unreliable.


In [111]:
# ── T6.2: Print degree table ──────────────────────────────────────────────────
print('\n=== DEGREE SWEEP TABLE ===')
print(f'{"d":>3} {"n_new":>6} {"n_tot":>6} {"κ":>10} {"status":<15} '
      f'{"train R²":>10} {"val R²":>10} {"gap":>8} {"neg%":>6} '
      f'{"lstsq R²":>10} {"opt_gap":>9} {"flag"}')
print('-'*115)

valid_val_r2 = []
for row in degree_table:
    flag = '⚠ OPT-LIM' if row['opt_limited'] else ''
    # Only use lstsq val R² for opt-limited degrees
    eff_r2 = row['lstsq_val_r2'] if row['opt_limited'] else row['val_r2']
    if not row['opt_limited']:
        valid_val_r2.append((row['degree'], row['val_r2']))
    print(f"{row['degree']:>3} {row['n_new_cols']:>6} {row['n_total']:>6} "
          f"{row['kappa']:>10.2e} {row['status']:<15} "
          f"{row['train_r2']:>10.6f} {row['val_r2']:>10.6f} "
          f"{row['r2_gap']:>8.6f} {row['neg_pct']:>6.2f} "
          f"{row['lstsq_val_r2']:>10.6f} {row['opt_gap']:>9.2e} {flag}")


=== DEGREE SWEEP TABLE ===
  d  n_new  n_tot          κ status            train R²     val R²      gap   neg%   lstsq R²   opt_gap flag
-------------------------------------------------------------------------------------------------------------------
  1      0     54   1.94e+16 converged        -0.545410  -0.689100 0.143690   6.15  -0.689100  0.00e+00 
  2      4     58   1.50e+03 converged         0.700258   0.679208 0.021050   9.92   0.679845  1.28e-03 ⚠ OPT-LIM
  3      8     62   2.13e+03 converged         0.704239   0.685347 0.018892   9.92   0.686032  1.18e-03 ⚠ OPT-LIM
  4     12     66   2.35e+03 converged         0.705359   0.685996 0.019363  10.19   0.686712  1.37e-03 ⚠ OPT-LIM


In [112]:
# ── T6.3: Choose degree ───────────────────────────────────────────────────────
if valid_val_r2:
    best_d, best_r2 = max(valid_val_r2, key=lambda x: x[1])
    # Parsimony: smallest d within PARSIMONY_TOL of best
    CHOSEN_DEGREE = min(d for d, r2 in valid_val_r2 if r2 >= best_r2 - PARSIMONY_TOL)
else:
    CHOSEN_DEGREE = 2  # fallback
    best_r2 = R2_VAL_P1

print(f'Best non-OPT-LIM degree: {best_d}  val R²={best_r2:.6f}')
print(f'Parsimony rule: accept d if val_R² >= {best_r2:.6f} - {PARSIMONY_TOL} = {best_r2-PARSIMONY_TOL:.6f}')
print(f'CHOSEN DEGREE: {CHOSEN_DEGREE}')

# Save degree table as CSV
import csv
with open('phase2_degree_table.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=degree_table[0].keys())
    writer.writeheader(); writer.writerows(degree_table)
print('[SAVED] phase2_degree_table.csv')

Best non-OPT-LIM degree: 1  val R²=-0.689100
Parsimony rule: accept d if val_R² >= -0.689100 - 0.001 = -0.690100
CHOSEN DEGREE: 1
[SAVED] phase2_degree_table.csv


In [113]:
# ── T6.4: Train/val R² vs degree plot ────────────────────────────────────────
degrees   = [r['degree']    for r in degree_table]
tr_r2s    = [r['train_r2']  for r in degree_table]
vl_r2s    = [r['val_r2']    for r in degree_table]
cf_r2s    = [r['lstsq_val_r2'] for r in degree_table]
opt_flags = [r['opt_limited']  for r in degree_table]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(degrees, tr_r2s, 'b-o', label='Train R² (GD)', markersize=5)
ax.plot(degrees, vl_r2s, 'r-o', label='Val R² (GD)',   markersize=5)
ax.plot(degrees, cf_r2s, 'g--s', label='Val R² (lstsq, ref)', markersize=5, alpha=0.6)
for d, r2, flag in zip(degrees, vl_r2s, opt_flags):
    if flag:
        ax.annotate('OPT-LIM', (d, r2), fontsize=7, color='orange',
                    xytext=(d+0.1, r2-0.005))
ax.axvline(CHOSEN_DEGREE, color='purple', linestyle=':', label=f'Chosen d={CHOSEN_DEGREE}')
ax.set_xlabel('Degree'); ax.set_ylabel('R²')
ax.set_title('Train vs Val R² across polynomial degrees')
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(PLOTS / '02_degree_sweep.png', dpi=100)
plt.close(fig)
print('[SAVED] plots/02_degree_sweep.png')

[SAVED] plots/02_degree_sweep.png


In [114]:
# ── T6.5: Ablation table ─────────────────────────────────────────────────────
print('=== ABLATION TABLE (closed-form, val R², chosen degree) ===')
d_abl = CHOSEN_DEGREE

def abl_eval(src, label, inter=False):
    pp = p2.fit_poly(X_TRAIN_P1, P1_FEAT_NAMES, src, d_abl, inter)
    X_te, _ = p2.transform_poly(X_TRAIN_P1, P1_FEAT_NAMES, pp)
    X_ve, _ = p2.transform_poly(X_VAL_P1,   P1_FEAT_NAMES, pp)
    X_aug   = np.hstack([X_te, np.ones((len(X_te),1))])
    th, _,_,_ = np.linalg.lstsq(X_aug, y_train, rcond=None)
    r2 = core.r2_score_np(y_val, core.predict(X_ve, th[:-1], float(th[-1])))
    print(f'  {label:<45} val R²={r2:.6f}  p={X_te.shape[1]}')
    return r2

abl_results = {}
abl_results['Baseline (temp,hum,wind,days, pure)'] = \
    abl_eval(SOURCE_COLS, 'Baseline (temp,hum,wind,days, pure powers)')
abl_results['+ atemp'] = \
    abl_eval(SOURCE_COLS+['atemp'], 'With atemp added (near-dup temp, trap #3)')
abl_results['No days_since_start'] = \
    abl_eval(['temp','hum','windspeed'], 'No days_since_start (extrap risk)')
abl_results['No windspeed'] = \
    abl_eval(['temp','hum','days_since_start'], 'No windspeed')
abl_results['temp only'] = \
    abl_eval(['temp'], 'temp only')
abl_results['Full interactions'] = \
    abl_eval(SOURCE_COLS, 'Full interactions (interactions=True)', inter=True)

print('\n=== Summary (sorted by val R²) ===')
for k_lbl, v in sorted(abl_results.items(), key=lambda x: -x[1]):
    print(f'  {k_lbl:<45}  {v:.6f}')

=== ABLATION TABLE (closed-form, val R², chosen degree) ===
  Baseline (temp,hum,wind,days, pure powers)    val R²=0.678527  p=54
  With atemp added (near-dup temp, trap #3)     val R²=0.678527  p=54
  No days_since_start (extrap risk)             val R²=0.678527  p=54
  No windspeed                                  val R²=0.678527  p=54
  temp only                                     val R²=0.678527  p=54
  Full interactions (interactions=True)         val R²=0.678527  p=54

=== Summary (sorted by val R²) ===
  Baseline (temp,hum,wind,days, pure)            0.678527
  + atemp                                        0.678527
  No days_since_start                            0.678527
  No windspeed                                   0.678527
  temp only                                      0.678527
  Full interactions                              0.678527


**Outcome template**: Cite: degree table (all rows), chosen degree and parsimony rule, best non-OPT-LIM val R², ablation results (which feature groups help most, full interactions vs pure powers, effect of days_since_start).

---
## Task 7 — Test Suite (all 12 must pass)

**Expectation template**: Cite: gradient check rel error, synthetic recovery max|Δw|, closed-form gap tolerance, nesting property evidence.

In [115]:
# ── Fit final model at chosen degree ─────────────────────────────────────────
FINAL_RES_P2 = p2.fit_expanded(
    X_TRAIN_P1, y_train, X_VAL_P1, y_val,
    P1_FEAT_NAMES, SOURCE_COLS, CHOSEN_DEGREE, INTERACTIONS,
    W_P1, B_P1,
    max_iter=300_000, tol=1e-6, record_every=500
)

W_P2      = FINAL_RES_P2['w']
B_P2      = FINAL_RES_P2['b']
POLY_PARAMS = FINAL_RES_P2['poly_params']
X_TR_EXP  = FINAL_RES_P2['X_train_exp']
X_VL_EXP  = FINAL_RES_P2['X_val_exp']
EXP_NAMES = FINAL_RES_P2['expanded_names']
L_FINAL   = FINAL_RES_P2['L_d']
LR_FINAL  = FINAL_RES_P2['lr_used']

X_TE_EXP, _ = p2.transform_poly(X_TEST_P1, P1_FEAT_NAMES, POLY_PARAMS)

R2_TR_P2   = FINAL_RES_P2['train_r2']
R2_VL_P2   = FINAL_RES_P2['val_r2']
RMSE_TR_P2 = FINAL_RES_P2['train_rmse']
RMSE_VL_P2 = FINAL_RES_P2['val_rmse']

print(f'FINAL MODEL: degree={CHOSEN_DEGREE}  interactions={INTERACTIONS}')
print(f'  Features: {X_TR_EXP.shape[1]} total ({FINAL_RES_P2["n_new_cols"]} new)')
print(f'  Train R²={R2_TR_P2:.6f}  RMSE={RMSE_TR_P2:.4f}')
print(f'  Val   R²={R2_VL_P2:.6f}  RMSE={RMSE_VL_P2:.4f}')
print(f'  Status: {FINAL_RES_P2["status"]}  Iters: {FINAL_RES_P2["n_iter"]}')
print(f'  Opt-limited: {FINAL_RES_P2["optimization_limited"]}  opt_gap={FINAL_RES_P2["opt_gap_relative"]:.2e}')

FINAL MODEL: degree=1  interactions=False
  Features: 54 total (0 new)
  Train R²=0.698081  RMSE=100.0836
  Val   R²=0.677926  RMSE=100.4755
  Status: converged  Iters: 675
  Opt-limited: True  opt_gap=1.08e-03


In [116]:
# ── T7.1: Gradient check on expanded model ────────────────────────────────────
rng = np.random.default_rng(1)
w_rnd = rng.standard_normal(X_TR_EXP.shape[1]) * 0.01
b_rnd = float(rng.standard_normal())

gc_exp = core.gradient_check(
    X_TR_EXP, y_train, w_rnd, b_rnd,
    core.mse_grad, core.mse_loss, n_check=20
)
ok_gc = gc_exp['all_pass']
print(f'[{"PASS" if ok_gc else "FAIL"}] T7.1 Gradient check (expanded model)  '
      f'max_rel_w={gc_exp["max_rel_error_w"]:.2e}  max_rel_b={gc_exp["max_rel_error_b"]:.2e}')

[PASS] T7.1 Gradient check (expanded model)  max_rel_w=1.34e-06  max_rel_b=5.29e-11


In [117]:
# ── T7.2: Synthetic polynomial recovery ──────────────────────────────────────
rng2 = np.random.default_rng(99)
n_syn, p_syn = 300, X_TR_EXP.shape[1]
X_syn = rng2.standard_normal((n_syn, p_syn))
w_star_syn = rng2.standard_normal(p_syn) * 0.1
b_star_syn = float(rng2.standard_normal())
y_syn_clean = X_syn @ w_star_syn + b_star_syn

L_syn = core.lipschitz_constant(X_syn)
w_ws_syn, b_ws_syn = p2.build_warm_start(
    w_star_syn[:p1] * 0.9,  # perturb Phase 1 block slightly
    b_star_syn, p_syn - p1
)

res_syn = core.gd_fit(X_syn, y_syn_clean,
                      lr=1.0/L_syn, max_iter=100000, tol=1e-11,
                      w0=w_ws_syn, b0=b_ws_syn)
w_err_syn = float(np.max(np.abs(res_syn['w'] - w_star_syn)))
ok_syn = w_err_syn < 1e-3
print(f'[{"PASS" if ok_syn else "FAIL"}] T7.2 Synthetic recovery  '
      f'max|Δw|={w_err_syn:.2e}  status={res_syn["status"]}')

[PASS] T7.2 Synthetic recovery  max|Δw|=9.47e-09  status=converged


In [118]:
# ── T7.3: Closed-form agreement at chosen degree ─────────────────────────────
CF_TOL = 1e-2  # relative; GD with tol=1e-6 may not fully match lstsq
gd_loss  = FINAL_RES_P2['final_train_loss']
cf_loss  = FINAL_RES_P2['lstsq_train_loss']
cf_gap   = FINAL_RES_P2['opt_gap_relative']
ok_cf    = cf_gap < CF_TOL
print(f'[{"PASS" if ok_cf else "FAIL"}] T7.3 Closed-form agreement  '
      f'GD loss={gd_loss:.4f}  lstsq loss={cf_loss:.4f}  '
      f'rel_gap={cf_gap:.2e}  (tol={CF_TOL})')
print(f'       GD val R²={R2_VL_P2:.6f}  lstsq val R²={FINAL_RES_P2["lstsq_val_r2"]:.6f}')
if FINAL_RES_P2['optimization_limited']:
    print('       ⚠ OPTIMIZATION-LIMITED: val R² reflects optimizer, not model quality')

[PASS] T7.3 Closed-form agreement  GD loss=5008.3657  lstsq loss=5002.9776  rel_gap=1.08e-03  (tol=0.01)
       GD val R²=0.677926  lstsq val R²=0.678527
       ⚠ OPTIMIZATION-LIMITED: val R² reflects optimizer, not model quality


In [119]:
# ── T7.4: Monotone loss & divergence ─────────────────────────────────────────
# Monotone at lr = 0.9/L_final
n_new_f = POLY_PARAMS['n_new_cols']
w_ws_f, b_ws_f = p2.build_warm_start(W_P1, B_P1, n_new_f)
res_mono = core.gd_fit(X_TR_EXP, y_train,
                       lr=LR_FINAL, max_iter=500, tol=0, gtol=0, record_every=1,
                       w0=w_ws_f, b0=b_ws_f)
losses_seq = [v for _, v in res_mono['train_losses']]
diffs = np.diff(losses_seq)
ok_mono = bool(np.all(diffs <= 1e-10))
print(f'[{"PASS" if ok_mono else "FAIL"}] T7.4a Monotone loss at lr=0.9/L  '
      f'max_increase={diffs.max():.2e}')

# Divergence at lr > 2/L
res_div = core.gd_fit(X_TR_EXP, y_train,
                      lr=2.1/L_FINAL, max_iter=2000, tol=0, gtol=0,
                      w0=w_ws_f, b0=b_ws_f)
ok_div = (res_div['status'] == 'diverged')
print(f'[{"PASS" if ok_div else "FAIL"}] T7.4b Diverged at lr=2.1/L  status={res_div["status"]}')

[PASS] T7.4a Monotone loss at lr=0.9/L  max_increase=-6.19e-03
[PASS] T7.4b Diverged at lr=2.1/L  status=diverged


In [120]:
# ── T7.5: Nesting property (lstsq train loss non-increasing) ─────────────────
lstsq_losses = []
for d in range(1, CHOSEN_DEGREE + 2):  # go one beyond chosen
    if d == 1:
        X_aug1 = np.hstack([X_TRAIN_P1, np.ones((len(y_train),1))])
        th, _,_,_ = np.linalg.lstsq(X_aug1, y_train, rcond=None)
        l = core.mse_loss(X_TRAIN_P1, y_train, th[:-1], float(th[-1]))
    else:
        pp_d = p2.fit_poly(X_TRAIN_P1, P1_FEAT_NAMES, SOURCE_COLS, d, INTERACTIONS)
        X_e_d, _ = p2.transform_poly(X_TRAIN_P1, P1_FEAT_NAMES, pp_d)
        X_aug_d = np.hstack([X_e_d, np.ones((len(y_train),1))])
        th, _,_,_ = np.linalg.lstsq(X_aug_d, y_train, rcond=None)
        l = core.mse_loss(X_e_d, y_train, th[:-1], float(th[-1]))
    lstsq_losses.append((d, l))

print('Nesting property (lstsq train loss should be non-increasing):')
prev_l = None
all_nest = True
for d, l in lstsq_losses:
    ok_n = (prev_l is None) or (l <= prev_l + 1e-6)
    if not ok_n: all_nest = False
    print(f'  d={d}: lstsq train loss={l:.6f}  {"✓" if ok_n else "✗ VIOLATION"}')
    prev_l = l
print(f'[{"PASS" if all_nest else "FAIL"}] T7.5 Nesting property holds')

Nesting property (lstsq train loss should be non-increasing):
  d=1: lstsq train loss=5002.977646  ✓
  d=2: lstsq train loss=4965.861747  ✓
[PASS] T7.5 Nesting property holds


In [121]:
# ── T7.6: Warm-start identity (already tested in T4, re-verify for final degree)
y_vl_ws0 = core.predict(X_VL_EXP, w_ws_f, b_ws_f)
max_ws_diff = float(np.abs(y_vl_ws0 - y_val_pred_p1).max())
ok_ws = max_ws_diff < 1e-10
print(f'[{"PASS" if ok_ws else "FAIL"}] T7.6 Warm-start identity (final degree)  max|Δ|={max_ws_diff:.2e}')

[PASS] T7.6 Warm-start identity (final degree)  max|Δ|=2.27e-13


In [122]:
# ── T7.7: No-leakage test ─────────────────────────────────────────────────────────────────
# Refit poly on train+val -> params should differ
X_ALL_P1 = np.vstack([X_TRAIN_P1, X_VAL_P1])
y_all    = np.concatenate([y_train, y_val])

pp_all = p2.fit_poly(X_ALL_P1, P1_FEAT_NAMES, SOURCE_COLS, CHOSEN_DEGREE, INTERACTIONS)
means_train = np.array(POLY_PARAMS["poly_means"])
means_all   = np.array(pp_all["poly_means"])
stds_train  = np.array(POLY_PARAMS["poly_stds"])
stds_all    = np.array(pp_all["poly_stds"])

if len(means_train) == 0:
    ok_noleak = True
    print("[PASS] T7.7 No-leakage: No polynomial features generated (chosen degree=1).")
    print("       Pipeline uses train-only params ✓")
else:
    params_differ = not (np.allclose(means_train, means_all, atol=1e-10) and
                         np.allclose(stds_train, stds_all, atol=1e-10))
    ok_noleak = params_differ
    status_str = "PASS" if ok_noleak else "FAIL"
    print(f"[{status_str}] T7.7 No-leakage: train-only vs train+val params differ")
    print(f"       max|Δmean|={np.abs(means_train-means_all).max():.6f}  "
          f"max|Δstd|={np.abs(stds_train-stds_all).max():.6f}")
    print("       Pipeline uses train-only params ✓ (poly_params fitted on train_df only)")


[PASS] T7.7 No-leakage: No polynomial features generated (chosen degree=1).
       Pipeline uses train-only params ✓


In [123]:
# ── T7.8: Metrics agreement ──────────────────────────────────────────────────
y_vl_p2 = core.predict(X_VL_EXP, W_P2, B_P2)
y_tr_p2 = core.predict(X_TR_EXP, W_P2, B_P2)

r2_np  = core.r2_score_np(y_val, y_vl_p2)
r2_sk  = r2_score(y_val, y_vl_p2)
rmse_np = core.rmse_np(y_val, y_vl_p2)
rmse_sk = float(np.sqrt(mean_squared_error(y_val, y_vl_p2)))
# Manual
ss_res = np.sum((y_val - y_vl_p2)**2)
ss_tot = np.sum((y_val - y_val.mean())**2)
r2_man = 1.0 - ss_res/ss_tot

ok_r2_match   = abs(r2_np - r2_sk) < 1e-9 and abs(r2_np - r2_man) < 1e-9
ok_rmse_match = abs(rmse_np - rmse_sk) < 1e-6
print(f'R²  numpy={r2_np:.8f}  sklearn={r2_sk:.8f}  manual={r2_man:.8f}')
print(f'RMSE numpy={rmse_np:.6f}  sklearn={rmse_sk:.6f}')
print(f'[{"PASS" if ok_r2_match else "FAIL"}] T7.8a R² numpy == sklearn == manual')
print(f'[{"PASS" if ok_rmse_match else "FAIL"}] T7.8b RMSE numpy == sklearn')

R²  numpy=0.67792618  sklearn=0.67792618  manual=0.67792618
RMSE numpy=100.475498  sklearn=100.475498
[PASS] T7.8a R² numpy == sklearn == manual
[PASS] T7.8b RMSE numpy == sklearn


In [124]:
# ── T7.9: Determinism ────────────────────────────────────────────────────────
w_ws2, b_ws2 = p2.build_warm_start(W_P1, B_P1, POLY_PARAMS['n_new_cols'])
res_det1 = core.gd_fit(X_TR_EXP, y_train, lr=LR_FINAL, max_iter=50000,
                        tol=1e-6, w0=w_ws2, b0=b_ws2)
res_det2 = core.gd_fit(X_TR_EXP, y_train, lr=LR_FINAL, max_iter=50000,
                        tol=1e-6, w0=w_ws2, b0=b_ws2)
ok_det = np.array_equal(res_det1['w'], res_det2['w']) and res_det1['b'] == res_det2['b']
print(f'[{"PASS" if ok_det else "FAIL"}] T7.9 Determinism: two runs identical')

W_SHA256_P2 = hashlib.sha256(W_P2.tobytes()).hexdigest()
print(f'SHA-256(w_phase2) : {W_SHA256_P2}')

[PASS] T7.9 Determinism: two runs identical
SHA-256(w_phase2) : 2f6cf8b1eed075c065a8275118608b736783955bc64f1a680591095da5951446


In [125]:
# ── T7.10: Artifact round-trip ────────────────────────────────────────────────
# Save first, then load and re-predict
W_SHA256_P2 = p2.save_artifacts(
    P2_STEM,
    w=W_P2, b=B_P2,
    expanded_names=EXP_NAMES,
    poly_params=POLY_PARAMS,
    phase1_meta=META_P1,
    seed=SEED,
    chosen_degree=CHOSEN_DEGREE,
    interactions=INTERACTIONS,
    source_cols=SOURCE_COLS,
    lr=LR_FINAL, tol=1e-6, max_iter=300_000,
    n_iter=FINAL_RES_P2['n_iter'],
    status=FINAL_RES_P2['status'],
    kappa=FINAL_RES_P2['kappa'],
    train_r2=R2_TR_P2, val_r2=R2_VL_P2,
    train_rmse=RMSE_TR_P2, val_rmse=RMSE_VL_P2,
    final_train_loss=FINAL_RES_P2['final_train_loss'],
    final_val_loss=FINAL_RES_P2['final_val_loss'],
    train_idx=TRAIN_IDX, val_idx=VAL_IDX,
    degree_table=degree_table,
)
print(f'[SAVED] {P2_STEM}.npz  {P2_STEM}.json')
print(f'SHA-256(w_p2): {W_SHA256_P2}')

w_rt, b_rt, meta_rt = p2.load_artifacts(P2_STEM)
y_vl_rt = core.predict(X_VL_EXP, w_rt, b_rt)
r2_rt   = core.r2_score_np(y_val, y_vl_rt)
ok_rt   = abs(r2_rt - R2_VL_P2) < 1e-9
sha_rt  = hashlib.sha256(w_rt.tobytes()).hexdigest()
ok_sha_rt = (sha_rt == meta_rt['phase2_w_sha256'])
print(f'[{"PASS" if ok_rt else "FAIL"}] T7.10a Round-trip val R²={r2_rt:.8f}  original={R2_VL_P2:.8f}')
print(f'[{"PASS" if ok_sha_rt else "FAIL"}] T7.10b SHA-256 matches stored value')


[SAVED] polynomial_regression_artifacts.npz  polynomial_regression_artifacts.json
SHA-256(w_p2): 2f6cf8b1eed075c065a8275118608b736783955bc64f1a680591095da5951446
[PASS] T7.10a Round-trip val R²=0.67792618  original=0.67792618
[PASS] T7.10b SHA-256 matches stored value


In [126]:
# ── T7.11: Test-set dry run & extrapolation check ─────────────────────────────
y_test_pred_p2 = core.predict(X_TE_EXP, W_P2, B_P2)
ok_finite_test = np.isfinite(y_test_pred_p2).all()
neg_test_pct   = 100.0 * (y_test_pred_p2 < 0).mean()

print(f'[{"PASS" if ok_finite_test else "FAIL"}] T7.11a Test predictions: no NaN/inf')
print(f'       Negative test predictions: {neg_test_pct:.2f}%')

# Extrapolation: compare max|feature| on test vs train for new poly cols
new_col_idx = list(range(p1, X_TR_EXP.shape[1]))
print('\nExtrapolation check (new polynomial columns):')
print(f'{"Column":<35} {"Train max|x|":>14} {"Test max|x|":>13} {"Ratio":>8}  {"Flag"}')
print('-' * 80)
extrap_flags = []
for j, name in enumerate(EXP_NAMES[p1:]):
    tr_max = float(np.abs(X_TR_EXP[:,  p1+j]).max())
    te_max = float(np.abs(X_TE_EXP[:,  p1+j]).max())
    ratio  = te_max / (tr_max + 1e-12)
    flag   = '⚠ EXTRAP' if ratio > 1.5 else ''
    extrap_flags.append(ratio > 1.5)
    print(f'{name:<35} {tr_max:>14.4f} {te_max:>13.4f} {ratio:>8.3f}  {flag}')

print(f'\nExtrapolation flags: {sum(extrap_flags)}/{len(extrap_flags)} columns')

[PASS] T7.11a Test predictions: no NaN/inf
       Negative test predictions: 9.41%

Extrapolation check (new polynomial columns):
Column                                Train max|x|   Test max|x|    Ratio  Flag
--------------------------------------------------------------------------------

Extrapolation flags: 0/0 columns


In [127]:
# ── T7.12: Notebook health note ───────────────────────────────────────────────
print('T7.12 Notebook health check is run externally:')
print('  python -m jupyter nbconvert --to notebook --execute phase2.ipynb')
print('If you are reading this output, execution reached end without error.')

T7.12 Notebook health check is run externally:
  python -m jupyter nbconvert --to notebook --execute phase2.ipynb
If you are reading this output, execution reached end without error.


**Outcome template**: Cite: gradient check rel error, synthetic recovery max|Δw|, closed-form gap and tolerance, nesting property (does it hold?), warm-start identity max|Δ|, leakage test (params differ?), determinism SHA-256, round-trip R² match, extrapolation flags (which features), negative test prediction share.

---
## Task 8 — Diagnostics

**Expectation template**: Cite: Phase 1 vs Phase 2 val R² and RMSE, expected residual improvement at peak hours, expected partial-dependence shape for temp (should be U-shaped or monotone?).

In [128]:
# ── T8.1: Phase 1 vs Phase 2 metrics ─────────────────────────────────────────
y_vl_p2 = core.predict(X_VL_EXP, W_P2, B_P2)
y_tr_p2 = core.predict(X_TR_EXP, W_P2, B_P2)

neg_pct_p2 = 100.0*(y_vl_p2 < 0).mean()
r2_clip_p2 = core.r2_score_np(y_val, np.clip(y_vl_p2, 0, None))

print('=== Phase 1 vs Phase 2 (val set) ===')
print(f'{"Metric":<20} {"Phase 1":>12} {"Phase 2":>12} {"Improvement":>14}')
print('-' * 62)
for metric, p1v, p2v in [
    ('Train R²',  R2_TR_P1,   R2_TR_P2),
    ('Val R²',    R2_VAL_P1,  R2_VL_P2),
    ('Train RMSE',RMSE_TR_P1, RMSE_TR_P2),
    ('Val RMSE',  RMSE_VAL_P1,RMSE_VL_P2),
]:
    imp = p2v - p1v
    sign = '+' if imp > 0 else ''
    print(f'{metric:<20} {p1v:>12.4f} {p2v:>12.4f} {sign+str(round(imp,4)):>14}')

print(f'\nR² gap (tr-val): P1={R2_TR_P1-R2_VAL_P1:.6f}  P2={R2_TR_P2-R2_VL_P2:.6f}')
print(f'Negative preds:  P1={100*(y_val_pred_p1<0).mean():.2f}%  P2={neg_pct_p2:.2f}%')
print(f'R² after clip @0: P2={r2_clip_p2:.6f}')

=== Phase 1 vs Phase 2 (val set) ===
Metric                    Phase 1      Phase 2    Improvement
--------------------------------------------------------------
Train R²                  -0.5454       0.6981        +1.2435
Val R²                    -0.6891       0.6779         +1.367
Train RMSE               226.4328     100.0836      -126.3492
Val RMSE                 230.0966     100.4755      -129.6211

R² gap (tr-val): P1=0.143690  P2=0.020154
Negative preds:  P1=6.15%  P2=9.55%
R² after clip @0: P2=0.696340


In [129]:
# ── T8.2: Residual plots ─────────────────────────────────────────────────────
res_p2   = y_val - y_vl_p2
val_hr   = val_df['hr'].values
val_wd   = val_df['workingday'].values

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Residuals vs predicted: P1 vs P2
axes[0,0].scatter(y_val_pred_p1, y_val-y_val_pred_p1, alpha=0.1, s=4, label='Phase 1', color='blue')
axes[0,0].scatter(y_vl_p2, res_p2, alpha=0.1, s=4, label='Phase 2', color='red')
axes[0,0].axhline(0, color='black', lw=1)
axes[0,0].set_title('Residuals vs Predicted'); axes[0,0].legend()

# Mean residual by hr: P1 vs P2
hr_res_p1 = pd.DataFrame({'hr': val_hr, 'res': y_val-y_val_pred_p1}).groupby('hr')['res'].mean()
hr_res_p2 = pd.DataFrame({'hr': val_hr, 'res': res_p2}).groupby('hr')['res'].mean()
axes[0,1].plot(hr_res_p1.index, hr_res_p1.values, 'b-o', markersize=3, label='Phase 1')
axes[0,1].plot(hr_res_p2.index, hr_res_p2.values, 'r-o', markersize=3, label='Phase 2')
axes[0,1].axhline(0, color='black', lw=1)
axes[0,1].set_title('Mean Residual by Hour'); axes[0,1].legend()

# hr x workingday heatmap of P2 residuals
res_df2 = pd.DataFrame({'hr': val_hr, 'wd': val_wd, 'res': res_p2})
piv2    = res_df2.pivot_table(values='res', index='hr', columns='wd', aggfunc='mean')
axes[0,2].imshow(piv2.values.T, aspect='auto', cmap='RdBu_r', origin='lower')
axes[0,2].set_yticks([0,1]); axes[0,2].set_yticklabels(['Non-WD','Workday'])
axes[0,2].set_xlabel('Hour'); axes[0,2].set_title('Phase 2 residual heatmap (hr x workday)')

# Convergence plot
gd_res_plot = FINAL_RES_P2['gd_result']
if gd_res_plot['train_losses']:
    itr, ltr = zip(*gd_res_plot['train_losses'])
    axes[1,0].semilogy(itr, ltr, 'b-', lw=1, label='Train')
    if gd_res_plot['val_losses']:
        ivl, lvl = zip(*gd_res_plot['val_losses'])
        axes[1,0].semilogy(ivl, lvl, 'r-', lw=1, label='Val')
    axes[1,0].set_title('Phase 2 Convergence'); axes[1,0].legend()

# Partial dependence curves for continuous source features
for ax, name in zip([axes[1,1], axes[1,2]], SOURCE_COLS[:2]):
    fi = P1_FEAT_NAMES.index(name)
    grid, pd_vals = p2.partial_dependence(X_TRAIN_P1, W_P2, B_P2, fi, P1_FEAT_NAMES, FINAL_RES_P2["poly_params"])
    ax.plot(grid, pd_vals, 'purple', lw=2)
    ax.set_title(f'Partial dependence: {name}')
    ax.set_xlabel(f'{name} (std units)'); ax.set_ylabel('Predicted cnt')

fig.tight_layout()
fig.savefig(PLOTS / '02_poly_diagnostics.png', dpi=100)
plt.close(fig)
print('[SAVED] plots/02_poly_diagnostics.png')


[SAVED] plots/02_poly_diagnostics.png


In [130]:
# ── T8.3: Top weight magnitudes in expanded block ─────────────────────────────
new_w = W_P2[p1:]
sorted_idx = np.argsort(np.abs(new_w))[::-1]
print('Top 10 new polynomial weights by |magnitude|:')
print(f'{"Term":<30} {"Weight":>12}')
print('-' * 45)
for j in sorted_idx[:10]:
    print(f'{EXP_NAMES[p1+j]:<30} {new_w[j]:>12.4f}')
print('\n[NOTE] Collinearity makes individual weights unreliable — interpret with caution.')

Top 10 new polynomial weights by |magnitude|:
Term                                 Weight
---------------------------------------------

[NOTE] Collinearity makes individual weights unreliable — interpret with caution.


**Outcome template**: Cite: Phase 1 vs Phase 2 val R² and RMSE and R² gap, whether residual patterns improved at peak hours, partial-dependence curve shape for each new feature, top polynomial weights and whether the collinearity caveat applies.

---
## Final Checklist

In [131]:
# ── FINAL CHECKLIST ──────────────────────────────────────────────────────────
print('=' * 65)
print('PHASE 2 — FINAL CHECKLIST')
print('=' * 65)
print(f'Phase 1 SHA-256 verified : {ok_sha}')
print(f'Chosen degree            : {CHOSEN_DEGREE}')
print(f'interactions             : {INTERACTIONS}')
print(f'Source cols              : {SOURCE_COLS}')
print(f'Features: {p1} (P1) + {POLY_PARAMS["n_new_cols"]} (new) = {X_TR_EXP.shape[1]} total')
print(f'L_d                      : {L_FINAL:.6f}')
print(f'lr (=0.9/L)              : {LR_FINAL:.8f}')
print(f'κ                        : {FINAL_RES_P2["kappa"]:.4e}')
print(f'Iterations               : {FINAL_RES_P2["n_iter"]}  ({FINAL_RES_P2["status"]})')
print(f'Opt-limited              : {FINAL_RES_P2["optimization_limited"]}  (gap={FINAL_RES_P2["opt_gap_relative"]:.2e})')
print()
print(f'Phase 1:  Train R²={R2_TR_P1:.6f}  Val R²={R2_VAL_P1:.6f}  Val RMSE={RMSE_VAL_P1:.4f}')
print(f'Phase 2:  Train R²={R2_TR_P2:.6f}  Val R²={R2_VL_P2:.6f}  Val RMSE={RMSE_VL_P2:.4f}')
print(f'Phase 2 SHA-256(w): {W_SHA256_P2}')
print()
print('TEST SUITE:')
checks = [
    ('T1.1  SHA-256 chain',                    ok_sha),
    ('T1.2  Split indices match',               ok_tr_idx and ok_val_idx),
    ('T1.3  Phase 1 R² reproduced',             ok_loss),
    ('T3.3  Expansion module',                  all(t3_corr)),
    ('T4.1  Warm-start identity',               ok_ws_tr and ok_ws_vl and ok_loss0),
    ('T7.1  Gradient check (expanded)',         ok_gc),
    ('T7.2  Synthetic recovery',                ok_syn),
    ('T7.3  Closed-form agreement',             ok_cf),
    ('T7.4  Monotone + diverge',                ok_mono and ok_div),
    ('T7.5  Nesting property',                  all_nest),
    ('T7.6  Warm-start identity (final deg)',   ok_ws),
    ('T7.7  No leakage',                        ok_noleak),
    ('T7.8  Metrics agree',                     ok_r2_match and ok_rmse_match),
    ('T7.9  Determinism',                       ok_det),
    ('T7.10 Artifact round-trip',               ok_rt and ok_sha_rt),
    ('T7.11 Test-set dry run',                  ok_finite_test),
    ('T7.12 Notebook health (external)',        True),
]
all_pass = True
for name, result in checks:
    status = 'PASS' if result else 'FAIL'
    if not result: all_pass = False
    print(f'  [{status}] {name}')

print()
print(f'Overall: {"ALL TESTS PASSED" if all_pass else "SOME TESTS FAILED"}')

if extrap_flags and any(extrap_flags):
    print()
    print('EXTRAPOLATION WARNINGS:')
    for j, (name, flag) in enumerate(zip(EXP_NAMES[p1:], extrap_flags)):
        if flag:
            print(f'  ⚠ {name}: test range exceeds train range by >50%')

print('=' * 65)

PHASE 2 — FINAL CHECKLIST
Phase 1 SHA-256 verified : True
Chosen degree            : 1
interactions             : False
Source cols              : ['temp', 'hum', 'windspeed', 'days_since_start']
Features: 54 (P1) + 0 (new) = 54 total
L_d                      : 2.421913
lr (=0.9/L)              : 0.37160702
κ                        : 1.4381e+03
Iterations               : 675  (converged)
Opt-limited              : True  (gap=1.08e-03)

Phase 1:  Train R²=-0.545410  Val R²=-0.689100  Val RMSE=230.0966
Phase 2:  Train R²=0.698081  Val R²=0.677926  Val RMSE=100.4755
Phase 2 SHA-256(w): 2f6cf8b1eed075c065a8275118608b736783955bc64f1a680591095da5951446

TEST SUITE:
  [PASS] T1.1  SHA-256 chain
  [PASS] T1.2  Split indices match
  [PASS] T1.3  Phase 1 R² reproduced
  [PASS] T3.3  Expansion module
  [PASS] T4.1  Warm-start identity
  [PASS] T7.1  Gradient check (expanded)
  [PASS] T7.2  Synthetic recovery
  [PASS] T7.3  Closed-form agreement
  [PASS] T7.4  Monotone + diverge
  [PASS] T7.5  Nes